In [ ]:
# 0. ติดตั้ง zstd สำหรับแตกไฟล์ติดตั้ง Ollama
!apt-get update -qq && apt-get install -y -qq zstd

# 1. ติดตั้ง Ollama สำหรับรันโมเดล AI (Gemma) ใน Colab
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 2. ดาวน์โหลดโมเดล Gemma 2 ขนาด 2B (เบา เร็ว และรองรับภาษาไทย)
!ollama pull gemma2:2b

# 3. ติดตั้งไลบรารีสำหรับ MQTT และ Telegram Bot
!pip install -q paho-mqtt python-telegram-bot nest_asyncio requests
print("=== ติดตั้ง AI และไลบรารีเสร็จสมบูรณ์ ===")

In [ ]:
import json
import requests
import nest_asyncio
import paho.mqtt.client as mqtt
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes

nest_asyncio.apply()

# ================= 1. ตั้งค่า Token และรหัสบอร์ด =================
TELEGRAM_TOKEN = "YOUR_TELEGRAM_BOT_TOKEN"
BOARD_ID       = "cucumber-room-01"  # ต้องตรงกับ BOARD_ID ในโค้ดบอร์ด

MQTT_BROKER     = "broker.emqx.io"
MQTT_PORT       = 1883
TOPIC_TELEMETRY = f"cucumber-aiot/{BOARD_ID}/telemetry"
TOPIC_COMMAND   = f"cucumber-aiot/{BOARD_ID}/command"

# เก็บค่าเซนเซอร์ล่าสุดที่บอร์ดส่งมา
latest_sensor = {
    "temp": None,
    "humidity": None,
    "pressure": None,
    "mode": "UNKNOWN",
    "led": "UNKNOWN"
}

# ================= 2. ระบบรับ-ส่งข้อมูล MQTT (ทำงานเบื้องหลัง) =================
def on_connect(client, userdata, flags, reason_code, properties):
    print(f"[MQTT] Connected! Subscribing to {TOPIC_TELEMETRY}")
    client.subscribe(TOPIC_TELEMETRY)

def on_message(client, userdata, msg):
    global latest_sensor
    try:
        data = json.loads(msg.payload.decode())
        latest_sensor.update(data)
    except Exception as e:
        print("[MQTT] Parse error:", e)

mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message
mqtt_client.connect(MQTT_BROKER, MQTT_PORT, 60)
mqtt_client.loop_start()

def send_board_command(cmd_name: str):
    payload = json.dumps({"cmd": cmd_name})
    mqtt_client.publish(TOPIC_COMMAND, payload)
    print(f"[MQTT] Sent Command -> {payload}")

# ================= 3. สมองกล AI (Gemma) แปลภาษาไทยเป็นคำสั่ง =================
def ask_gemma(user_text: str) -> dict:
    sensor_context = json.dumps(latest_sensor, ensure_ascii=False)

    system_prompt = f"""คุณคือผู้ช่วย AI ดูแลห้องอัจฉริยะที่เชื่อมต่อกับบอร์ด IoT (Cucumber ESP32-S2)
ข้อมูลเซนเซอร์ในห้องตอนนี้คือ: {sensor_context}

หน้าที่ของคุณคือวิเคราะห์ข้อความของผู้ใช้ แล้วตอบกลับเป็น JSON เท่านั้น โดยมี 2 คีย์คือ "intent" และ "reply"
รายการ "intent" ที่เลือกใช้ได้มีดังนี้:
- "GET_STATUS" : ผู้ใช้ถามอุณหภูมิ ความชื้น ความดันอากาศ สภาพห้อง หรือสถานะไฟปัจจุบัน
- "LED_RED"    : ผู้ใช้สั่งเปิดไฟสีแดง หรือเปลี่ยนไฟเป็นสีแดง
- "LED_GREEN"  : ผู้ใช้สั่งเปิดไฟสีเขียว หรือเปลี่ยนไฟเป็นสีเขียว
- "LED_BLUE"   : ผู้ใช้สั่งเปิดไฟสีฟ้า หรือสีน้ำเงิน
- "LED_OFF"    : ผู้ใช้สั่งปิดไฟ
- "MODE_AUTO"  : ผู้ใช้สั่งให้กลับไปทำงานโหมดอัตโนมัติ (auto)
- "CHAT"       : ทักทาย หรือถามความรู้ทั่วไปเกี่ยวกับ IoT

กติกา:
1. ในช่อง "reply" ให้ตอบเป็นภาษาไทยที่สุภาพ เป็นกันเอง กระชับ และอ้างอิงตัวเลขเซนเซอร์จริงเมื่อถูกถาม
2. หากค่าเซนเซอร์ยังเป็น null ให้แจ้งผู้ใช้ว่ายังไม่ได้รับข้อมูลจากบอร์ด
3. ตอบเป็น JSON ที่ถูกต้องเท่านั้น ห้ามมีข้อความอื่นนอกปีกกา"""

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "gemma2:2b",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            "format": "json",
            "stream": False
        }
    )

    raw_content = response.json()["message"]["content"]
    return json.loads(raw_content)

# ================= 4. ส่วนจัดการแชต Telegram Bot =================
async def start_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    welcome = (
        "สวัสดีครับ! ผมคือผู้ช่วย AI ประจำบอร์ด Cucumber\n"
        "ลองพิมพ์คุยกับผมได้เลย เช่น:\n"
        "• ตอนนี้ในห้องอุณหภูมิเท่าไร ร้อนไหม?\n"
        "• ช่วยเปิดไฟสีฟ้าหน่อย\n"
        "• ปิดไฟให้ที\n"
        "• กลับไปโหมดอัตโนมัติ"
    )
    await update.message.reply_text(welcome)

async def handle_chat(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_msg = update.message.text
    print(f"[Telegram] User: {user_msg}")

    try:
        result = ask_gemma(user_msg)
        intent = result.get("intent", "CHAT")
        reply  = result.get("reply", "ขออภัยครับ ผมไม่เข้าใจคำสั่ง")

        # ถ้า AI ตีความว่าเป็นคำสั่งควบคุมบอร์ด ให้ส่ง MQTT ทันที
        valid_commands = ["LED_RED", "LED_GREEN", "LED_BLUE", "LED_OFF", "MODE_AUTO"]
        if intent in valid_commands:
            send_board_command(intent)

        await update.message.reply_text(reply)
    except Exception as e:
        print("[Error]", e)
        await update.message.reply_text("เกิดข้อผิดพลาดในการประมวลผล AI กรุณาลองใหม่อีกครั้งครับ")

# เริ่มรันบอท
app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
app.add_handler(CommandHandler("start", start_cmd))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_chat))

print("=== Telegram Bot & AI พร้อมทำงานแล้ว! ลองทักแชตใน Telegram ได้เลย ===")
app.run_polling()